# Equity Total Return Swaps in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Portfolio and transactions |
| 4 | Valuation |
| 5 | Maturity as an event |

## The instrument

A total return swap has two legs pulling in opposite directions: one side carries the return on
the referenced **index or stock** (the asset leg), and the other pays the cost of financing that
exposure (the funding leg):

    asset leg     the referenced index, as an Equity
    funding leg   a FloatingLeg paying an index plus a spread

## The asset leg

An `Equity` carries no coupon or maturity of its own -- it is referenced inline via
`m.Equity(instrument_type="Equity", dom_ccy=..., identifiers=m.EquityAllOfIdentifiers(client_internal=...))`
purely to name the underlying. All of the TRS's own economics -- start, maturity, reset level,
financing -- live on the swap and its funding leg instead.

---
## Setup

In [1]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

# Looks for a secrets file at FBN_SECRETS_PATH, then secrets.json in this folder; falls back to
# LUSID's own environment-variable config (FBN_LUSID_URL, FBN_TOKEN_URL, FBN_USERNAME,
# FBN_PASSWORD, FBN_CLIENT_ID, FBN_CLIENT_SECRET, FBN_APP_NAME) if neither is present. See
# secrets.example.json for the file's shape.
SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

`SimpleStatic` prices the position from the quoted mark alone.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "EquityTrsDemo"
RECIPE    = "equity-trs-demo-recipe"
PORTFOLIO = "equity-trs-demo-book"

SWAP_ID   = "DEMO-EQTRS-01"
INDEX_ID  = "DEMO-TECH100"
DESC      = "Demo Tech 100 Index TRS"
CURRENCY  = "USD"
START     = d(2025, 5, 1)
MATURITY  = d(2026, 5, 1)
ASOF      = d(2025, 11, 15)

FINANCING_INDEX      = "SOFRRATE"
FINANCING_FIXING_REF = "USD-SOFR"
FINANCING_SPREAD     = 0.0125       # added on top of the fixing reference
FINANCING_DAY_COUNT  = "Actual360"
ASSET_SIDE   = "Receive"
FUNDING_SIDE = "Pay"

INITIAL_LEVEL   = 4_500.00          # index level at inception, for the asset leg's own record
NOTIONAL        = 1.0               # funding leg notional, kept at a per-unit value -- the
                                     # position's own quantity carries the actual size
RESET_FREQUENCY = "3M"              # reset frequency for the asset leg's reset schedule

QUANTITY = 200.00
PRICE    = 625.00                   # quoted mark, per unit

print(f"{DESC}")
print(f"  asset   {ASSET_SIDE:<8} {INDEX_ID} at index level {INITIAL_LEVEL:,.2f}")
print(f"  funding {FUNDING_SIDE:<8} {FINANCING_INDEX} + {FINANCING_SPREAD:.2%} ({FINANCING_FIXING_REF})")
print(f"  {QUANTITY:,.0f} units at {PRICE:,.2f} {CURRENCY} on {ASOF:%Y-%m-%d}")
print(f"  market value = {QUANTITY:,.0f} x {PRICE:,.2f} = {QUANTITY * PRICE:,.2f} {CURRENCY}")

Demo Tech 100 Index TRS
  asset   Receive  DEMO-TECH100 at index level 4,500.00
  funding Pay      SOFRRATE + 1.25% (USD-SOFR)
  200 units at 625.00 USD on 2025-11-15
  market value = 200 x 625.00 = 125,000.00 USD


---
# 1. Instrument creation

The asset leg's `Equity` just names the referenced index -- it doesn't carry any conventions of
its own. The index level at inception lives on the swap's own `initial_price` instead.

In [3]:
asset = m.Equity(
    instrument_type="Equity",
    dom_ccy=CURRENCY,
    identifiers=m.EquityAllOfIdentifiers(client_internal=INDEX_ID))

trs = m.TotalReturnSwap(
    instrument_type="TotalReturnSwap",
    start_date=START,
    maturity_date=MATURITY,
    asset_leg=m.AssetLeg(
        asset=asset,
        pay_receive=ASSET_SIDE,
        initial_price=INITIAL_LEVEL,
        reset_schedule=m.ResetSchedule(frequency=RESET_FREQUENCY),
        income_policy="PassThrough"),
    funding_leg=m.FloatingLeg(
        instrument_type="FloatingLeg",
        start_date=START,
        maturity_date=MATURITY,
        notional=NOTIONAL,
        leg_definition=m.LegDefinition(
            rate_or_spread=FINANCING_SPREAD,
            pay_receive=FUNDING_SIDE,
            conventions=m.FlowConventions(
                currency=CURRENCY,
                payment_frequency=RESET_FREQUENCY,
                day_count_convention=FINANCING_DAY_COUNT,
                roll_convention=str(START.day),
                payment_calendars=[], reset_calendars=[]),
            index_convention=m.IndexConvention(
                currency=CURRENCY,
                payment_tenor="1D",
                fixing_reference=FINANCING_FIXING_REF,
                index_name=FINANCING_INDEX,
                publication_day_lag=0,
                day_count_convention=FINANCING_DAY_COUNT),
            reset_convention="InArrears",
            stub_type="ShortBack",
            notional_exchange_type="None")))

TRS_LUID = upsert("trs", DESC, SWAP_ID, trs)
print(f"Equity TRS : {TRS_LUID}")

Equity TRS : LUID_00003DFS


---
# 2. Recipe

Under `SimpleStatic`, the position is valued straight off its quoted mark. The legs' conventions
describe how the instrument is structured, but under this model, they don't feed into the
valuation itself.

In [4]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Equity TRS, marked",
            market=m.MarketContext(
                market_rules=[m.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                    quote_type="Price", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_instrument_code_type="LusidInstrumentId",
                    default_scope=SCOPE)),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="SimpleStatic",
                    instrument_type="TotalReturnSwap")],
                options=m.PricingOptions(allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: EquityTrsDemo/equity-trs-demo-recipe


---
# 3. Portfolio and transactions

Striking a TRS doesn't exchange any principal, so `totalConsideration` is zero.

In [5]:
recreate_portfolio(PORTFOLIO, "Equity TRS Demo Book", CURRENCY, d(2025, 1, 1), recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-TRS",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": TRS_LUID},
        transaction_date=START.isoformat(),
        settlement_date=START.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=0.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=0.0, currency=CURRENCY),
        source="default")])

display(transactions(PORTFOLIO, START, START))

Recreated EquityTrsDemo/equity-trs-demo-book


,date,type,luid,units,consideration
0,2025-05-01,Buy,LUID_00003DFS,200.00,0.00


---
# 4. Valuation

Just one quote is needed: the swap's own price, per unit.

In [6]:
upsert_price(TRS_LUID, PRICE, ASOF, CURRENCY)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/CleanPV",       "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, CURRENCY)
display(result)

pv = result.loc[result["Instrument/default/Name"] == DESC, "Sum(Valuation/CleanPV)"].iloc[0]
print(f"LUSID CleanPV {pv:,.2f}  vs  quoted mark {QUANTITY * PRICE:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/CleanPV)
0,Demo Tech 100 Index TRS,200.00,"125,000.00"


LUSID CleanPV 125,000.00  vs  quoted mark 125,000.00


---
# 5. Maturity as an event

A portfolio's `instrumentEventConfiguration` has to point at a recipe, and it can only be set at
creation time -- that's what passing `recipe=RECIPE` into `recreate_portfolio()` back in section 3
took care of. Skip it and `query_applicable_instrument_events` just comes back empty, with no error
to explain why.

The `TotalReturnSwap` doesn't need anything extra here: `MaturityEvent` is applicable by default
and comes with a populated transaction. Query a window that runs past `MATURITY` and it comes back
directly -- the swap's end date isn't just a static field, it's a forecastable event.

In [7]:
events_api = api(lusid.InstrumentEventsApi)

WINDOW_END = MATURITY + timedelta(days=5)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=ASOF.isoformat(),
        window_end=WINDOW_END.isoformat(),
        effective_at=WINDOW_END.isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

,event type,eligible balance,status
0,MaturityEvent,200.00,Active


---
# Summary

1. An equity TRS is a `TotalReturnSwap` whose asset leg is an `Equity` -- there just to name the
   underlying index or stock, with no conventions of its own.
2. The index level at inception sits on the swap's own `initial_price`. The funding leg's
   `notional` stays at 1.0, since it's a per-unit contract -- it's the position's own quantity
   that scales it up to size, not the notional.
3. Under `SimpleStatic`, the position is valued straight off its quoted mark; the legs'
   conventions describe the instrument, but they don't drive the valuation.
4. `MaturityEvent` only becomes forecastable once the portfolio's own `instrumentEventConfiguration`
   points at a recipe -- set that at creation via `recreate_portfolio(..., recipe=RECIPE)` and a
   `TotalReturnSwap` needs nothing more.

In [8]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Instrument : {TRS_LUID}")

Scope      : EquityTrsDemo
Portfolio  : EquityTrsDemo/equity-trs-demo-book
Recipe     : EquityTrsDemo/equity-trs-demo-recipe
Instrument : LUID_00003DFS
